## 1. Setup

In [ ]:
import importlib, subprocess, sys
for pkg in ["emoji", "transformers"]:
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import re, string, random, unicodedata, collections, warnings, copy, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import emoji
from scipy import sparse
from scipy.stats import chi2_contingency
from sklearn.model_selection import train_test_split
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline, make_pipeline, make_union
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (accuracy_score, f1_score, precision_recall_fscore_support,
                             classification_report, confusion_matrix)

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 200)
SEED = 42
random.seed(SEED); np.random.seed(SEED)
sns.set_theme(style="whitegrid")

In [ ]:
# ---- Output folders and save helpers: everything this notebook produces is written under ./outputs ----
import json, joblib
from pathlib import Path

OUT = Path("outputs")
DIRS = {k: OUT / k for k in ["data", "results", "figures", "models", "predictions", "reports"]}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

def slug(name):
    return re.sub(r"[^A-Za-z0-9]+", "_", str(name)).strip("_").lower()

def save(df_, name, sub="results"):
    """Save a DataFrame as UTF-8 CSV (opens correctly in Excel) in outputs/<sub>/."""
    df_.to_csv(DIRS[sub] / name, index=False, encoding="utf-8-sig")
    print(f"saved {DIRS[sub] / name}  ({len(df_)} rows)")

def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o))
    print(f"saved {path}")

def save_fig(name, fig=None, show=True):
    """Save the current (or given) figure as a 200-dpi PNG in outputs/figures/, then show it."""
    fig = fig or plt.gcf()
    fig.tight_layout()
    fig.savefig(DIRS["figures"] / f"{name}.png", dpi=200, bbox_inches="tight")
    print(f"saved {DIRS['figures'] / (name + '.png')}")
    if show: plt.show()

def save_model(obj, name):
    """Save a fitted sklearn pipeline with cloudpickle, which stores the notebook-defined classes (Prep, CueFeatures ...)
    inside the file, so it reloads in a fresh session with plain pickle.load (needs sklearn + emoji installed)."""
    from joblib.externals import cloudpickle
    p = DIRS["models"] / f"{slug(name)}.pkl"
    with open(p, "wb") as f:
        cloudpickle.dump(obj, f)
    print(f"saved {p}  ({p.stat().st_size/1e6:.1f} MB)")

## 2. Load data and drop missing labels

In [ ]:
from pathlib import Path
FILE_PATH = "dataset.csv"
CSV = Path(FILE_PATH)
if not CSV.exists():
    try:                                   # Google Colab: upload the file when asked
        from google.colab import files
        up = files.upload()
        CSV = Path(next(iter(up)))
    except ImportError:
        raise FileNotFoundError("Put dataset.csv next to this notebook.")

raw = pd.read_csv(CSV, encoding="utf-8-sig")
raw = raw.loc[:, ~raw.columns.str.startswith("Unnamed")]      # drop the empty trailing-comma columns
print("Raw shape:", raw.shape)

cols = {c.lower().strip(): c for c in raw.columns}
TEXT_COL  = cols.get("urdu_text") or cols.get("urdu text")
LABEL_COL = cols.get("is_sarcastic") or cols.get("is_scaristic")
assert TEXT_COL and LABEL_COL, f"Unexpected columns: {list(raw.columns)}"

data = raw[[TEXT_COL, LABEL_COL]].rename(columns={TEXT_COL: "text", LABEL_COL: "label"})
print("Missing text :", data.text.isna().sum())
print("Missing label:", data.label.isna().sum())
print("Label values (raw):", data.label.value_counts(dropna=False).to_dict())

data["label"] = pd.to_numeric(data["label"], errors="coerce")
data = data.dropna(subset=["text", "label"])
data = data[data.label.isin([0, 1])].copy()
data["label"] = data.label.astype(int)
data["text"] = data.text.astype(str)
print("After dropping missing/invalid labels:", data.shape)

## 3. Exploratory data analysis

### 3.1 Class distribution

In [ ]:
counts = data.label.value_counts().sort_index()
pct = (counts / counts.sum() * 100).round(2)


ax = counts.plot(kind="bar", color=["skyblue", "salmon"], figsize=(4.5, 3.5))
ax.set_xticklabels(["Non-sarcastic (0)", "Sarcastic (1)"], rotation=0)
save(pd.DataFrame({"label": counts.index, "count": counts.values, "percent": pct.values}), "eda_class_distribution_raw.csv")
ax.set_ylabel("Posts"); ax.set_title("Class distribution")
ax.bar_label(ax.containers[0]); save_fig("01_class_distribution_raw")

### 3.2 Text-length statistics

In [ ]:
def emojis_in(s):
    return [e["emoji"] for e in emoji.emoji_list(s)]

data["n_chars"]  = data.text.str.len()
data["n_tokens"] = data.text.str.split().str.len()
data["n_emoji"]  = data.text.apply(lambda s: len(emojis_in(s)))

stats = (data.groupby("label")[["n_chars", "n_tokens", "n_emoji"]]
             .describe(percentiles=[.25, .5, .75, .95]).T.round(2))
print(stats)
save(stats.reset_index().rename(columns={"level_0": "feature", "level_1": "stat", 0: "non_sarcastic", 1: "sarcastic"}),
     "eda_length_stats.csv")

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for col, ax in zip(["n_chars", "n_tokens"], axes[:2]):
    d = data[data[col] <= data[col].quantile(0.99)]
    sns.histplot(data=d, x=col, hue="label", bins=50, stat="density",
                 common_norm=False, element="step", ax=ax)
    ax.set_title(f"{col} by class (clipped at 99th pct)")
sns.boxplot(data=data[data.n_tokens <= data.n_tokens.quantile(.99)], x="label", y="n_tokens", ax=axes[2])
axes[2].set_title("Tokens per post by class")
save_fig("02_text_length_distributions")
print("95th percentile of tokens:", int(data.n_tokens.quantile(.95)),
      "| 99th:", int(data.n_tokens.quantile(.99)))

After Data distribution


- Class balance: The dataset has 9,993 non-sarcastic and 9,962 sarcastic posts, so the classes are almost perfectly balanced.
- Sarcastic posts are longer: The median is 21 tokens (97 characters) for sarcastic posts and 13 tokens (60 characters) for non-sarcastic posts.
- Whole distribution is shifted: The middle 50% of posts spans 15 to 28 tokens for sarcastic and 7 to 21 for non-sarcastic. The spread is similar in both classes (std about 12 tokens), so this is not caused by a few long posts.
- Emoji counts are similar: The mean is 2.20 emojis per post for non-sarcastic and 1.91 for sarcastic, with the same median range. The number of emojis alone is therefore a weak cue, so which emojis appear is what needs testing.
- Outliers are rare: A few posts contain up to 71 emojis, but 95% of posts have 6 or fewer.
- Modelling implication: 95% of posts are under about 51 tokens, so a maximum sequence length of 64 tokens keeps almost all text.
- Limitation: The length gap may act as a dataset shortcut, because a model could partly learn that longer posts are sarcastic instead of learning real sarcasm cues.
- Balance after cleaning: the 50/50 balance above is for the raw rows only. Sarcastic posts are repeated far more often (3,567 of 4,689 repeat copies), so after de-duplication and label voting the modelling set is about 59% non-sarcastic / 41% sarcastic (8,651 vs 6,004 texts). This is why macro-F1, not accuracy, is the main metric.


### 3.3 Representative examples

In [ ]:
def show(sub, title, n=6, seed=SEED):
    print(f"\n===== {title} =====")
    for t in sub.sample(min(n, len(sub)), random_state=seed).text:
        print("-", t.replace("\n", " | ")[:250])

for lab, name in [(0, "NON-SARCASTIC"), (1, "SARCASTIC")]:
    sub = data[data.label == lab]
    show(sub, f"{name}: random", 8)
    show(sub[sub.n_tokens <= 4], f"{name}: very short", 4)
    show(sub[sub.n_tokens >= sub.n_tokens.quantile(.95)], f"{name}: very long", 3)
    show(sub[sub.n_emoji >= 2], f"{name}: emoji-heavy", 4)

### 3.4 Data-quality audit (missing values, duplicates, noise, script mix)

In [ ]:
def is_urdu_char(ch):
    o = ord(ch)
    return (0x0600 <= o <= 0x06FF or 0x0750 <= o <= 0x077F or
            0xFB50 <= o <= 0xFDFF or 0xFE70 <= o <= 0xFEFF)

def script_ratios(s):
    em = set("".join(emojis_in(s)))
    ns = [c for c in s if not c.isspace()]
    n = max(len(ns), 1)
    return pd.Series({
        "urdu_ratio":  sum(is_urdu_char(c) for c in ns) / n,
        "latin_ratio": sum(c.isascii() and c.isalpha() for c in ns) / n,
        "other_ratio": sum(1 for c in ns if not (c.isascii() or is_urdu_char(c) or c in em)) / n})

data = pd.concat([data, data.text.apply(script_ratios)], axis=1)

QUOTE_RE = re.compile(r"[\"'`“”‘’«»]")
checks = {
    "has_emoji":    data.n_emoji > 0,
    "has_quote":    data.text.str.contains(QUOTE_RE),
    "has_excl_or_q": data.text.str.contains(r"[!؟?]"),
    "repeat_char":  data.text.str.contains(r"([^\W\d_])\1{2,}"),
    "repeat_punct": data.text.str.contains(r"([!?؟۔.])\1+"),
    "has_hashtag":  data.text.str.contains(r"#\w+"),
    "has_mention":  data.text.str.contains(r"@\w+"),
    "has_url":      data.text.str.contains(r"https?://|www\."),
    "has_latin":    data.latin_ratio > 0,
    "has_newline":  data.text.str.contains("\n"),
    "foreign_chars>10%": data.other_ratio > 0.10,
    "very_short(<3 tok)": data.n_tokens < 3,
}
q = pd.DataFrame({k: {"overall_%": v.mean()*100,
                      "non-sarc_%": v[data.label == 0].mean()*100,
                      "sarc_%": v[data.label == 1].mean()*100} for k, v in checks.items()}).T.round(2)
print(q)
save(q.reset_index().rename(columns={"index": "check"}), "eda_quality_audit.csv")

dup = data.duplicated("text", keep=False)
print("\nExact duplicate rows:", int(dup.sum()), f"({dup.mean()*100:.2f}%)")
print("Duplicate texts with CONFLICTING labels:",
      int((data[dup].groupby("text").label.nunique() > 1).sum()))
print("Empty / whitespace-only texts:", int((data.text.str.strip() == "").sum()))

- Duplicates: 7,801 rows (39.09%) are exact duplicates. These were removed before splitting so the same post cannot appear in both train and test.
- Conflicting labels: 645 duplicate texts carry both labels. They are annotation noise, so they were dropped.
- Empty texts: none, and a few rows with missing labels were dropped.
- Corrupted characters: only 0.54% of posts have more than 10% foreign characters, so encoding damage is negligible. The garbled text in the assignment PDF is an export artifact.
- Hashtags, @mentions, URLs and line breaks: all are much more frequent in sarcastic posts (for example, hashtags in 7.32% of sarcastic vs 0.27% of non-sarcastic posts), so they were kept as cues.
- Code-mixing: Latin-script words appear in 20.32% of sarcastic posts vs 7.70% of non-sarcastic posts, so English and Roman tokens were kept, not removed.
- Emojis: present in 83% of posts overall, and more common in non-sarcastic posts (88.05% vs 77.85%). Their usefulness depends on which emoji appears, so they are tested in the ablation.
- Quotes and ! or ؟: slightly more common in sarcastic posts (5.16% vs 3.20%, and 6.89% vs 3.71%).
- Repeated punctuation: about the same in both classes (14.16% vs 13.70%), so it is a weak cue.
- Very short posts (under 3 tokens): almost never sarcastic (0.15% vs 4.57%).

- Limitation: features such as line breaks, hashtags, URLs and post length may reflect how the data was collected instead of real sarcasm, so models could learn these shortcuts.

In [ ]:
from sklearn.metrics import roc_auc_score

extra_cols = [c for c in raw.columns if c not in (TEXT_COL, LABEL_COL)]
print("Extra columns in the raw file:", extra_cols)
if extra_cols:
    display(raw[extra_cols].head(3))

miss = raw[raw[TEXT_COL].isna()]
print("\nMissing text by class:", miss[LABEL_COL].value_counts().to_dict())

vc = data.text.value_counts()
grp = data.groupby("text").label.agg(size="size", n_labels="nunique", sarc_share="mean")
print("\nUnique texts:", len(vc), "of", len(data), "rows")
print("Texts that appear more than once:", int((vc > 1).sum()))
print("Group-size distribution (how many copies of one text):")
print(grp["size"].value_counts().sort_index().head(10).to_string())
print("\nMost repeated texts:")
for t, n in vc.head(8).items():
    print(f"  x{n}: {t[:90]!r}")

dup_rows = data.duplicated("text", keep="first")
print("\nRows that are repeat copies (would be removed), by class:",
      data[dup_rows].label.value_counts().to_dict())
conf = grp[grp.n_labels > 1].sort_values("size", ascending=False)
print("Conflicting-label texts:", len(conf), "| median copies:", conf["size"].median())
print("Examples of conflicting texts (copies, share labelled sarcastic):")
for t, r in conf.head(6).iterrows():
    print(f"  x{int(r['size'])}, sarc={r.sarc_share:.2f}: {t[:90]!r}")

# do the extra columns explain the duplicates (e.g. same text from different sources)?
if extra_cols:
    rd = raw.dropna(subset=[TEXT_COL])
    g = rd.groupby(TEXT_COL)
    sizes = g.size()
    for c in extra_cols:
        varies = (g[c].nunique(dropna=False) > 1)[sizes > 1].mean()
        print(f"extra column {c!r}: differs inside {varies*100:.1f}% of duplicated texts")

In [ ]:
TATWEEL = "\u0640"
CTRL_RE = re.compile("[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]")
ZWC_RE  = re.compile("[\u200b-\u200f\u202a-\u202e\u2060\ufeff]")
noise_checks = {
    "tatweel/kashida (رحمـــتہ)":  data.text.str.contains(TATWEEL),
    "zero-width / bidi marks":     data.text.apply(lambda t: bool(ZWC_RE.search(t))),
    "control characters":          data.text.apply(lambda t: bool(CTRL_RE.search(t))),
    "curly / double-backtick quotes": data.text.str.contains(r"[“”«»„]|``|''"),
    "starts with @mention (reply)": data.text.str.contains(r"^\s*@\w+"),
    "ends with URL":               data.text.str.contains(r"https?://\S+\s*$"),
    "underscore hashtag w/o #":    data.text.str.contains(r"[^\W\d_]\w*_[^\W_]"),
    "trailing Latin fragment":     data.text.apply(lambda t: bool(re.search(
                                      r"[\u0600-\u06FF].*\s[A-Za-z][A-Za-z0-9]{2,}\s*$", t.replace("\n", " ")))),
    "contains digits":             data.text.str.contains(r"\d"),
    "ALL-CAPS Latin word":         data.text.str.contains(r"\b[A-Z]{3,}\b"),
}
nq = pd.DataFrame({k: {"overall_%": v.mean()*100, "non-sarc_%": v[data.label == 0].mean()*100,
                       "sarc_%": v[data.label == 1].mean()*100} for k, v in noise_checks.items()}).T.round(2)
display(nq)
save(nq.reset_index().rename(columns={"index": "check"}), "eda_noise_audit.csv")

In [ ]:
LAUGH_EMOJIS = "😂🤣😹😆😅😁😜😝🤪😏"
DIAC       = re.compile("[\u064B-\u065F\u0670\u06D6-\u06ED]")
LETTER_MAP = str.maketrans({"ي": "ی", "ى": "ی", "ك": "ک", "ه": "ہ", "ۃ": "ہ", "ة": "ہ", "أ": "ا", "إ": "ا"})
DIGIT_MAP  = str.maketrans("٠١٢٣٤٥٦٧٨٩۰۱۲۳۴۵۶۷۸۹", "0123456789" * 2)
URL_RE     = re.compile(r"https?://\S+|www\.\S+")
MENTION_RE = re.compile(r"@\w+")
HASHTAG_RE = re.compile(r"#(\w+)")
ELONG_RE   = re.compile(r"([^\W\d_])\1{2,}")
LAUGH_RE   = re.compile(r"(?:ہا){3,}|(?:ہی){3,}|(?:ha){3,}", re.I)
PREP_RE    = re.compile(r"([!?؟۔.،,])\1+")        # repeated punctuation run

def _clean_segment(seg):
    seg = unicodedata.normalize("NFKC", seg)
    seg = ZWC_RE.sub("", seg); seg = CTRL_RE.sub("", seg)
    seg = seg.replace(TATWEEL, "")                              # kashida stretching
    seg = re.sub(r"``|''|[“”«»„]", '"', seg)                     # one quote symbol
    seg = URL_RE.sub(" urltok ", seg)
    seg = MENTION_RE.sub(" usertok ", seg)
    seg = HASHTAG_RE.sub(r" hashtok \1 ", seg)                  # keep the tag word + a hashtag marker
    seg = seg.replace("\n", " nltok ").replace("\r", " ")       # keep "was multi-line" as a token
    seg = seg.replace("_", " ")                                  # لفافوں_کی_فیک_نیوز -> separate words
    seg = DIAC.sub("", seg.translate(LETTER_MAP).translate(DIGIT_MAP))
    seg = ELONG_RE.sub(r"\1\1", seg)                            # واااہ -> وااہ (flag kept separately)
    seg = PREP_RE.sub(r"\1", seg)                               # ۔۔۔ / !!! -> single mark (flag kept separately)
    return seg.lower()

def clean_text(s):
    # emojis are protected: only the text between them is normalized, emojis are kept and spaced
    out, last = [], 0
    for e in emoji.emoji_list(s):
        out.append(_clean_segment(s[last:e["match_start"]]))
        out.append(" " + e["emoji"].replace("\ufe0f", "") + " ")
        last = e["match_end"]
    out.append(_clean_segment(s[last:]))
    return re.sub(r"\s+", " ", "".join(out)).strip()

def cue_flags(s):
    # cue flags are read from the RAW text so that cleaning can never erase a cue
    s = s.astype(str); s_nt = s.str.replace(TATWEEL, "", regex=False)
    f = pd.DataFrame(index=s.index)
    f["n_emoji"]         = s.apply(lambda t: len(emoji.emoji_list(t)))
    f["has_emoji"]       = f.n_emoji > 0
    f["laugh_emoji"]     = s.apply(lambda t: any(c in LAUGH_EMOJIS for c in t))
    f["ends_with_emoji"] = s.apply(lambda t: (lambda el: bool(el) and el[-1]["match_end"] >= len(t.rstrip()))(emoji.emoji_list(t)))
    f["has_quote"]       = s.str.contains(QUOTE_RE)
    f["has_exclaim"]     = s.str.contains("!", regex=False)
    f["has_question"]    = s.str.contains(r"[؟?]")
    f["has_hashtag"]     = s.str.contains(r"#\w+")
    f["has_mention"]     = s.str.contains(r"@\w+")
    f["starts_with_mention"] = s.str.contains(r"^\s*@\w+")
    f["has_url"]         = s.str.contains(r"https?://|www\.")
    f["has_newline"]     = s.str.contains("\n", regex=False)
    f["has_latin"]       = s.str.contains(r"[A-Za-z]")
    f["has_elongation"]  = s_nt.str.contains(ELONG_RE)
    f["has_laughter"]    = s_nt.str.contains(LAUGH_RE)
    f["has_repeat_punct"] = s.str.contains(PREP_RE)
    f["very_short"]      = s.str.split().str.len() < 3
    return f

demo_rows = data.sample(6, random_state=7).text.tolist()
for t in demo_rows:
    print("RAW  :", t[:150].replace("\n", " | "))
    print("CLEAN:", clean_text(t)[:150], "\n")

Adding Tag of starting and ending this will help the model to distingiush the staring and ending


In [ ]:
# ---- settings -------------------------------------------------------------------
VOTE_HI, VOTE_LO = 0.75, 0.25                          # >=HI -> sarcastic, <=LO -> non-sarcastic, else ambiguous
START_TAG, END_TAG = "startok", "endtok"               # plain words so the tokenizer never splits them

# ---- 1. clean every row (raw index is kept as the row id) -------------------------
c = data[["text", "label"]].copy()
flags_all = cue_flags(c.text)                          # cue flags are read from the RAW text
c["text_clean"] = c.text.apply(clean_text)
c["key"] = (c.text_clean.str.replace(r"\b(?:urltok|usertok|nltok)\b", " ", regex=True)
                        .str.replace(r"\s+", " ", regex=True).str.strip())

log = [("raw file rows", len(raw)), ("after dropping missing text / label", len(c))]
dropped = [raw[raw[TEXT_COL].isna() | raw[LABEL_COL].isna()]
             .assign(raw_row_id=lambda d: d.index, reason="missing text or label",
                     text=lambda d: d[TEXT_COL], label=lambda d: d[LABEL_COL])
             [["raw_row_id", "reason", "text", "label"]]]

empty = c[c.key.str.len() == 0]
dropped.append(empty.assign(raw_row_id=empty.index, reason="empty once URLs/mentions are ignored")
                    [["raw_row_id", "reason", "text", "label"]])
c = c[c.key.str.len() > 0]
log.append(("after dropping texts that are empty once URLs/mentions are ignored", len(c)))

# ---- 2. one row per text: vote on the label inside every group of copies ----------
votes = c.groupby("key").label.agg(n_copies="size", n_sarc="sum")
votes["n_non"] = votes.n_copies - votes.n_sarc
votes["soft_label"] = votes.n_sarc / votes.n_copies          # share of copies labelled sarcastic
votes["label_confidence"] = np.select(
    [votes.n_sarc.eq(0) | votes.n_sarc.eq(votes.n_copies),
     votes.soft_label.ge(VOTE_HI) | votes.soft_label.le(VOTE_LO)],
    ["consistent", "majority_vote"], default="ambiguous")
votes["label_final"] = np.where(votes.label_confidence.eq("ambiguous"), np.nan,
                                (votes.soft_label >= 0.5).astype(float))

first = c.drop_duplicates("key").copy()                       # first copy represents the text
dups = c[c.duplicated("key", keep="first")]
dropped.append(dups.assign(raw_row_id=dups.index, reason="repeat copy (one row per text is kept)")
                   [["raw_row_id", "reason", "text", "label"]])
first = first.join(votes, on="key").rename(columns={"label": "label_original"})
first["raw_row_id"] = first.index
first["label"] = first.label_final
first["text_tagged"] = f"{START_TAG} " + first.text_clean + f" {END_TAG}"
first = first.join(flags_all)
first["n_tokens"] = first.text.str.split().str.len()
first["n_chars"]  = first.text.str.len()

lead = ["raw_row_id", "text", "text_clean", "text_tagged", "label", "label_original", "soft_label",
        "label_confidence", "n_copies", "n_sarc", "n_non"]
first = first[lead + [k for k in first.columns if k not in lead + ["key", "label_final"]]]

clean     = first[first.label_confidence != "ambiguous"].copy()      # hard labels: use for train / val / test
ambiguous = first[first.label_confidence == "ambiguous"].copy()      # soft labels only: never score on these
clean["label"] = clean.label.astype(int)
clean = clean.reset_index(drop=True)

log.append(("after keeping one row per text (copies collapsed)", len(first)))
log.append(("after setting aside ambiguous texts (soft label only)", len(clean)))
drop_log = pd.DataFrame(log, columns=["step", "rows"])
drop_log["removed"] = -drop_log.rows.diff().fillna(0).astype(int)
print(drop_log.to_string(index=False))

# ---- 3. what did the vote do? ------------------------------------------------------
print("\nTexts by label confidence:", first.label_confidence.value_counts().to_dict())
maj = first[first.label_confidence == "majority_vote"]
print("Majority-vote texts:", len(maj), "| label changed vs. the first copy:",
      int((maj.label != maj.label_original).sum()))
print("Ambiguous texts:", len(ambiguous), "| soft-label mean: %.2f" % ambiguous.soft_label.mean() if len(ambiguous) else "")
print("\nClean labelled set:", clean.shape, "| class counts:", clean.label.value_counts().sort_index().to_dict(),
      "| sarcastic share = %.3f" % clean.label.mean())

# ---- 4. save every intermediate file -----------------------------------------------
dup_groups = (first[first.n_copies > 1]
              .sort_values("n_copies", ascending=False)
              [["raw_row_id", "text", "n_copies", "n_sarc", "n_non", "soft_label", "label_confidence", "label"]])
save(drop_log, "drop_log.csv")
save(pd.concat(dropped), "dropped_rows.csv", sub="data")
save(dup_groups, "duplicate_groups.csv", sub="data")
save(first, "clean_all.csv", sub="data")                  # every unique text, incl. ambiguous
save(clean, "clean_labeled.csv", sub="data")              # the cleaned dataset (hard labels only)
save(ambiguous, "ambiguous.csv", sub="data")

In [ ]:
cols_cmp = ["has_emoji", "laugh_emoji", "has_quote", "has_exclaim", "has_question", "has_hashtag",
            "has_mention", "has_url", "has_newline", "has_latin", "has_elongation", "has_laughter",
            "has_repeat_punct", "very_short"]
y_all = data.label.loc[flags_all.index].values
rows = []
for k in cols_cmp:
    rows.append({"cue": k,
                 "before non-sarc %": flags_all.loc[y_all == 0, k].mean()*100,
                 "before sarc %":     flags_all.loc[y_all == 1, k].mean()*100,
                 "after non-sarc %":  clean.loc[clean.label == 0, k].mean()*100,
                 "after sarc %":      clean.loc[clean.label == 1, k].mean()*100})
cmp_tab = pd.DataFrame(rows).set_index("cue").round(2)
display(cmp_tab)

print("\nPost length (tokens) after cleaning:")
print(clean.groupby("label").n_tokens.describe(percentiles=[.5, .95]).round(2))

In [ ]:
# AUC 0.5 = no information; far from 0.5 = a possible dataset shortcut (not necessarily real sarcasm)
y = clean.label.values
auc = {k: roc_auc_score(y, clean[k].astype(float)) for k in
       ["n_tokens", "n_chars", "n_emoji"] + cols_cmp if clean[k].nunique() > 1}
auc = pd.Series(auc).round(3)
shortcut = pd.DataFrame({"AUC_alone": auc, "distance_from_0.5": (auc - 0.5).abs().round(3)})
display(shortcut.sort_values("distance_from_0.5", ascending=False))

In [ ]:
save(cmp_tab.reset_index(), "cue_before_after.csv")
save(shortcut.reset_index().rename(columns={"index": "feature"}), "cue_auc.csv")

## 4. De-duplication and stratified train/validation/test split

Duplicates (and near-duplicates that differ only by spacing/case/URLs/mentions) are removed **before** splitting so the same post can never appear in two splits.
Texts that appear with conflicting labels are dropped: they are annotation noise and cannot be learned.

In [ ]:
# ZW is still used by the Prep class later on, so keep this line (the old light_key / drop-conflicts block is not needed any more)
ZW = re.compile("[\u200b-\u200f\u202a-\u202e\u2060\ufeff\ufe0f]")

# `clean` (one row per text, hard labels, made by the cleaning cell) is split once, here.
train_clean, tmp_clean = train_test_split(clean, test_size=0.30, stratify=clean.label, random_state=SEED)
val_clean, test_clean  = train_test_split(tmp_clean, test_size=0.50, stratify=tmp_clean.label, random_state=SEED)
train_clean, val_clean, test_clean = [x.reset_index(drop=True) for x in (train_clean, val_clean, test_clean)]

for n, d_ in [("train", train_clean), ("val", val_clean), ("test", test_clean)]:
    print(f"{n:5}: {len(d_):6d} rows | sarcastic share = {d_.label.mean():.3f} | median tokens = {d_.n_tokens.median():.0f}")

keys = [set(x.text_clean) for x in (train_clean, val_clean, test_clean)]
assert not (keys[0] & keys[1]) and not (keys[0] & keys[2]) and not (keys[1] & keys[2]), "text leaked between splits"
print("No cleaned text appears in more than one split.")

save(train_clean, "train.csv", sub="data"); save(val_clean, "val.csv", sub="data"); save(test_clean, "test.csv", sub="data")
save(pd.DataFrame([{"split": n, "rows": len(d_), "sarcastic": int(d_.label.sum()), "non_sarcastic": int((d_.label == 0).sum()),
                    "sarcastic_share": round(d_.label.mean(), 4), "median_tokens": float(d_.n_tokens.median())}
                   for n, d_ in [("train", train_clean), ("val", val_clean), ("test", test_clean)]]), "split_summary.csv")

# Aliases so the modelling cells written earlier (tr_df, X_tr, y_tr ...) use THIS split, not a second one.
# The Prep class cleans raw text itself, so the models take the raw `text` column.
# (Switch to "text_clean" if you want the models to see the already-cleaned text.)
TEXT_FOR_MODELS = "text"
tr_df, va_df, te_df = train_clean, val_clean, test_clean
X_tr, y_tr = tr_df[TEXT_FOR_MODELS].tolist(), tr_df.label.values
X_va, y_va = va_df[TEXT_FOR_MODELS].tolist(), va_df.label.values
X_te, y_te = te_df[TEXT_FOR_MODELS].tolist(), te_df.label.values

## 5. Urdu-aware preprocessing and cue analysis

The `Prep` class below is a configurable pipeline, so every step can be switched on or off in the ablation (section 8).

| Step | What it does | Why it suits Urdu social-media text |
|---|---|---|
| NFKC + zero-width / bidi mark removal | maps presentation forms to the base letter, deletes invisible marks | the same word can be stored with different code points and look identical |
| Letter unification (ي→ی, ك→ک, ه/ۃ/ة→ہ, أ/إ→ا) | one code point per Urdu letter | Arabic- and Urdu-keyboard spellings of one word become one token |
| Diacritic stripping and digit unification | removes optional marks; ٠١٢ / ۰۱۲ → 012 | tweets use diacritics inconsistently |
| URL / @mention → `urltok` / `usertok` | replaces the string, keeps a marker | the identity is noise, but "has a link / is a reply" is a measured cue |
| Elongation and laughter flags (`elongtok`, `laughtok`, `punctreptok`) | collapses واااہ → وااہ and adds a flag token | the spelling is normalized but the expressive cue is kept |
| Emojis: keep / remove / group by sentiment | emojis stay as separate tokens by default | emojis are tested as a cue, not assumed useful or useless |
| Latin (code-mixed) words: keep / remove | kept by default | English words are about 3x more frequent in sarcastic posts |
| Hashtags: keep / drop | `#word` keeps the word | the hashtag text often carries the joke |

In [ ]:
DIAC       = re.compile("[\u064B-\u065F\u0670\u06D6-\u06ED]")
LETTER_MAP = str.maketrans({"ي": "ی", "ى": "ی", "ك": "ک", "ه": "ہ", "ۃ": "ہ", "ة": "ہ", "أ": "ا", "إ": "ا"})
DIGIT_MAP  = str.maketrans("٠١٢٣٤٥٦٧٨٩۰۱۲۳۴۵۶۷۸۹", "0123456789" * 2)
URL        = re.compile(r"https?://\S+|www\.\S+")
MENTION    = re.compile(r"@\w+")
HASHTAG    = re.compile(r"#\w+")
LATIN      = re.compile(r"[A-Za-z]+")
ELONG      = re.compile(r"([^\W\d_])\1{2,}")                  # a letter repeated 3+ times
LAUGH      = re.compile(r"(?:ہا){3,}|(?:ہی){3,}|(?:ha){3,}", re.I)
PUNCT_REP  = re.compile(r"([!?؟۔.])\1+")
PUNCT_RE   = re.compile("[" + re.escape(string.punctuation + "،؛؟۔٪٫٬«»…“”‘’") + "]")

EMO_GROUPS = [
    ("emounamused", ["unamused", "rolling_eyes", "expressionless", "neutral", "thinking", "smirk"]),
    ("emolaugh",    ["joy", "rofl", "rolling_on", "laugh", "grin", "wink", "tongue", "zany", "squint", "beaming"]),
    ("emosad",      ["cry", "sob", "pensive", "disappointed", "broken", "worried", "sad", "weary", "tired", "pleading", "frown"]),
    ("emolove",     ["heart", "kiss", "love", "hug"]),
    ("emoangry",    ["angry", "rage", "pout", "cursing", "swear", "enraged"]),
    ("emogesture",  ["thumbs", "pray", "folded", "clap", "fist", "hand", "ok", "victory", "muscle"]),
]
def emo_group(e):
    name = emoji.demojize(e).strip(":").lower()
    for g, keys in EMO_GROUPS:
        if any(k in name for k in keys):
            return g
    return "emoother"

class Prep:
    # picklable preprocessing callable
    # emo: keep | remove | group     punct: keep | remove
    # hashtag: keep | drop           latin: keep | remove
    def __init__(self, normalize=True, elong=True, emo="keep", punct="keep", hashtag="keep", latin="keep"):
        self.normalize, self.elong, self.emo = normalize, elong, emo
        self.punct, self.hashtag, self.latin = punct, hashtag, latin
    def __call__(self, s):
        s = unicodedata.normalize("NFKC", str(s))
        s = ZW.sub("", s)
        s = re.sub(r"``|''|[“”«»„]", '"', s)
        s = URL.sub(" urltok ", s)
        s = MENTION.sub(" usertok ", s)
        if self.hashtag == "drop":
            s = HASHTAG.sub(" ", s)
        if self.normalize:
            s = DIAC.sub("", s.translate(LETTER_MAP).translate(DIGIT_MAP))
        flags = []
        if self.elong:
            if LAUGH.search(s): flags.append("laughtok")
            if ELONG.search(s): flags.append("elongtok")
            if self.punct == "keep" and PUNCT_REP.search(s): flags.append("punctreptok")
            s = ELONG.sub(r"\1\1", s)
        s = LATIN.sub(" ", s) if self.latin == "remove" else s.lower()
        if self.punct == "remove":
            s = PUNCT_RE.sub(" ", s)
        if self.emo == "remove":
            s = emoji.replace_emoji(s, " ")
        elif self.emo == "group":
            s = emoji.replace_emoji(s, lambda e, dd: f" {emo_group(e)} ")
        else:
            s = emoji.replace_emoji(s, lambda e, dd: f" {e} ")
        return re.sub(r"\s+", " ", s + " " + " ".join(flags)).strip()

def make_prep(**kw):
    return Prep(**kw)

# word tokens that keep emojis / punctuation as separate tokens
KEEP = r"[\w\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06ED]+|[^\w\s]"
TOKEN_RE = re.compile(KEEP)

demo = make_prep()
for t in tr_df.sample(5, random_state=3).text:
    print("RAW :", t[:140].replace("\n", " | "))
    print("PREP:", demo(t)[:140], "\n")

In [ ]:
# ---- Do the engineered columns matter? (train for fitting, validation for judging; the test set is NOT touched) ----
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.inspection import permutation_importance
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import roc_auc_score, f1_score

# Columns made from the TEXT only. Never use n_copies, n_sarc, n_non, soft_label or label_confidence:
# they are built from the labels / from how often a text repeats, so they would leak the answer.
FEATURES = ["n_chars", "n_tokens", "n_emoji", "has_emoji", "laugh_emoji", "ends_with_emoji", "has_quote",
            "has_exclaim", "has_question", "has_hashtag", "has_mention", "starts_with_mention", "has_url",
            "has_newline", "has_latin", "has_elongation", "has_laughter", "has_repeat_punct", "very_short"]
GROUPS = {
    "length":                     ["n_chars", "n_tokens", "very_short"],
    "emoji":                      ["n_emoji", "has_emoji", "laugh_emoji", "ends_with_emoji"],
    "tweet structure":            ["has_hashtag", "has_mention", "starts_with_mention", "has_url", "has_newline"],
    "code-mixing (Latin)":        ["has_latin"],
    "punctuation":                ["has_quote", "has_exclaim", "has_question", "has_repeat_punct"],
    "expressive spelling":        ["has_elongation", "has_laughter"],
}
Ftr = train_clean[FEATURES].astype(float); Fva = val_clean[FEATURES].astype(float)
ytr, yva = train_clean.label.values, val_clean.label.values
w_tr = np.where(ytr == 1, (ytr == 0).sum() / (ytr == 1).sum(), 1.0)       # balance the classes

def gbm(): return HistGradientBoostingClassifier(max_depth=3, max_iter=200, learning_rate=0.05,
                                                 l2_regularization=1.0, random_state=SEED)
def fit_eval(cols, Xa=None, Xb=None):
    A = Ftr[cols] if Xa is None else Xa; B = Fva[cols] if Xb is None else Xb
    m = gbm().fit(A, ytr, sample_weight=w_tr)
    p = m.predict_proba(B)[:, 1]
    return m, p, roc_auc_score(yva, p), f1_score(yva, (p >= 0.5).astype(int), average="macro")

# 1) one column at a time
uni = pd.DataFrame({
    "AUC_train": {f: roc_auc_score(ytr, Ftr[f]) for f in FEATURES},
    "AUC_val":   {f: roc_auc_score(yva, Fva[f]) for f in FEATURES},
    "mutual_info": dict(zip(FEATURES, mutual_info_classif(Ftr, ytr, random_state=SEED))),
    "coverage_%": {f: (Ftr[f] > 0).mean() * 100 for f in FEATURES}}).round(3)
uni["strength"] = (uni.AUC_train - 0.5).abs().round(3)
print("1) Each column alone (AUC 0.5 = no information):")
display(uni.sort_values("strength", ascending=False))

# 2) groups of columns, alone and left out
rows = []
_, _, a, f = fit_eval(FEATURES); rows.append({"columns used": "ALL columns", "val_AUC": a, "val_macroF1": f})
for g, cols in GROUPS.items():
    _, _, a, f = fit_eval(cols);                       rows.append({"columns used": f"only {g}", "val_AUC": a, "val_macroF1": f})
for g, cols in GROUPS.items():
    rest = [c for c in FEATURES if c not in cols]
    _, _, a, f = fit_eval(rest);                       rows.append({"columns used": f"all except {g}", "val_AUC": a, "val_macroF1": f})
print("\n2) Groups of columns (gradient-boosted trees on the columns only, no text):")
grp_tab = pd.DataFrame(rows).round(4); display(grp_tab); save(grp_tab, "feature_check_groups.csv")

# 3) permutation importance: how much does validation AUC fall when one column is shuffled?
m_all, p_cols, _, _ = fit_eval(FEATURES)
pi = permutation_importance(m_all, Fva, yva, scoring="roc_auc", n_repeats=10, random_state=SEED)
pimp = pd.DataFrame({"AUC_drop": pi.importances_mean, "std": pi.importances_std}, index=FEATURES).round(4)
print("\n3) Permutation importance on validation:")
display(pimp.sort_values("AUC_drop", ascending=False))

# 4) do the columns add anything BEYOND the text?  text score (out-of-fold on train) +/- columns
txt_model = make_pipeline(
    TfidfVectorizer(preprocessor=make_prep(normalize=True, elong=True), token_pattern=KEEP,
                    ngram_range=(1, 2), min_df=2, sublinear_tf=True),
    LogisticRegression(C=3, max_iter=3000))
oof = cross_val_predict(txt_model, X_tr, y_tr, cv=StratifiedKFold(5, shuffle=True, random_state=SEED),
                        method="predict_proba")[:, 1]
txt_model.fit(X_tr, y_tr); p_txt_va = txt_model.predict_proba(X_va)[:, 1]
Str = Ftr.assign(text_score=oof); Sva = Fva.assign(text_score=p_txt_va)
rows = []
for name, cols in [("text score only", ["text_score"]),
                   ("text score + all columns", ["text_score"] + FEATURES),
                   ("text score + length only", ["text_score", "n_chars", "n_tokens"]),
                   ("columns only (no text)", FEATURES)]:
    _, _, a, f = fit_eval(cols, Str[cols], Sva[cols]); rows.append({"inputs": name, "val_AUC": a, "val_macroF1": f})
print("\n4) Do the columns add information beyond the text?")
beyond_tab = pd.DataFrame(rows).round(4); display(beyond_tab); save(beyond_tab, "feature_check_beyond_text.csv")

# 5) is the text model just reading length?  AUC of the text score inside length quartiles
q = pd.qcut(val_clean.n_tokens, 4, duplicates="drop")
rows = []
for b in q.cat.categories:
    idx = (q == b).values
    if len(set(yva[idx])) > 1:
        rows.append({"tokens": str(b), "n": int(idx.sum()), "sarcastic_share": yva[idx].mean().round(3),
                     "AUC_text_score": roc_auc_score(yva[idx], p_txt_va[idx]),
                     "AUC_length": roc_auc_score(yva[idx], val_clean.n_tokens.values[idx])})
print("\n5) Same-length comparison (does the text still separate the classes at a fixed length?):")
len_tab = pd.DataFrame(rows).round(3); display(len_tab); save(len_tab, "feature_check_same_length.csv")

pimp.sort_values("AUC_drop").plot.barh(y="AUC_drop", legend=False, figsize=(6, 5))
plt.xlabel("validation AUC lost when the column is shuffled"); plt.title("Which columns matter?"); save_fig("03_feature_permutation_importance")
save(uni.reset_index().rename(columns={"index": "column"}), "feature_check_single_columns.csv")
save(pimp.reset_index().rename(columns={"index": "column"}), "feature_check_permutation.csv")

In [ ]:
# ---- Label leakage test: posts whose hashtag/words NAME the label (#sarcasm, #sarcastic, #طنز ...) ----
LEAK_RE = re.compile(r"#?\S*(?:sarcas|satir|طنز)\S*", re.I)
def has_leak(texts): return np.array([bool(LEAK_RE.search(t)) for t in texts])
def mask_leak(t): return LEAK_RE.sub(" ", t)

ltr, lva = has_leak(X_tr), has_leak(X_va)
print("Posts containing a label-naming word/hashtag (train):")
print(f"  sarcastic: {ltr[y_tr == 1].mean()*100:.2f}%   non-sarcastic: {ltr[y_tr == 0].mean()*100:.2f}%")
print(f"  P(sarcastic | such a word) = {y_tr[ltr].mean():.3f}   (n={ltr.sum()})")
h = train_clean.has_hashtag.values
print(f"  hashtag posts WITHOUT such a word: P(sarcastic) = {y_tr[h & ~ltr].mean():.3f}   (n={(h & ~ltr).sum()})")
print("\nExamples:")
for t in np.array(X_tr, dtype=object)[ltr][:4]: print("  -", t[:120].replace("\n", " | "))

def word_lr():
    return make_pipeline(TfidfVectorizer(preprocessor=make_prep(normalize=True, elong=True), token_pattern=KEEP,
                                         ngram_range=(1, 2), min_df=2, sublinear_tf=True),
                         LogisticRegression(C=3, max_iter=3000))
m_orig = word_lr().fit(X_tr, y_tr)
m_mask = word_lr().fit([mask_leak(t) for t in X_tr], y_tr)
rows = [
 {"model / data": "trained on original text, all validation posts",           "val_macroF1": f1_score(y_va, m_orig.predict(X_va), average="macro")},
 {"model / data": "trained on original text, validation posts WITHOUT leak words",
  "val_macroF1": f1_score(y_va[~lva], m_orig.predict(np.array(X_va, dtype=object)[~lva].tolist()), average="macro")},
 {"model / data": "trained AND tested with leak words masked",
  "val_macroF1": f1_score(y_va, m_mask.predict([mask_leak(t) for t in X_va]), average="macro")}]
leak_tab = pd.DataFrame(rows).round(4); display(leak_tab); save(leak_tab, "label_leakage_test.csv")

### 5.1 Are the "noisy" cues actually informative? (train only)
Lift = P(sarcastic | cue present) / P(sarcastic | cue absent). A chi-square test checks whether the association is real.
This is the evidence for **keeping** a cue rather than stripping it.

In [ ]:
# ---- 5.1 lift + chi-square for every cue (TRAIN only) ----
rows = []
for k in cols_cmp:
    present = train_clean[k].astype(bool).values
    if present.all() or (~present).all():
        continue
    p1, p0 = y_tr[present].mean(), y_tr[~present].mean()
    chi2, pval, _, _ = chi2_contingency(pd.crosstab(present, y_tr))
    rows.append({"cue": k, "n_with_cue": int(present.sum()), "coverage_%": present.mean() * 100,
                 "P(sarc | cue)": p1, "P(sarc | no cue)": p0, "lift": p1 / p0, "chi2": chi2, "p_value": pval})
lift_tab = pd.DataFrame(rows).sort_values("lift", ascending=False).round(4).reset_index(drop=True)
display(lift_tab)
save(lift_tab, "cue_lift_chi2.csv")

## 6. Evaluation helpers

In [ ]:
RESULTS, VAL, PRED, PROB = {}, {}, {}, {}

def metrics(y_true, y_pred):
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1], zero_division=0)
    return {"accuracy": accuracy_score(y_true, y_pred),
            "P_0": p[0], "R_0": r[0], "F1_0": f[0],
            "P_1": p[1], "R_1": r[1], "F1_1": f[1],
            "macro_F1": f1_score(y_true, y_pred, average="macro")}

def score_proba(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    z = model.decision_function(X)
    return 1 / (1 + np.exp(-z))

def log_test(name, y_pred, val_f1, prob=None):
    # evaluate once on the TEST set, store everything for the results table
    RESULTS[name] = metrics(y_te, y_pred); VAL[name] = val_f1
    PRED[name] = np.asarray(y_pred)
    if prob is not None: PROB[name] = np.asarray(prob)
    print(f"[{name}] TEST macro-F1 = {RESULTS[name]['macro_F1']:.4f} | acc = {RESULTS[name]['accuracy']:.4f}")
    print(classification_report(y_te, y_pred, target_names=["non-sarcastic", "sarcastic"], digits=4))
    # ---- saved for every model: classification report (json) and confusion matrix (csv) ----
    save_json(classification_report(y_te, y_pred, target_names=["non-sarcastic", "sarcastic"], digits=4, output_dict=True),
              DIRS["reports"] / f"{slug(name)}_classification_report.json")
    pd.DataFrame(confusion_matrix(y_te, y_pred, labels=[0, 1]), index=["true_non_sarc", "true_sarc"],
                 columns=["pred_non_sarc", "pred_sarc"]).to_csv(DIRS["reports"] / f"{slug(name)}_confusion_matrix.csv")

def val_f1(model):
    return f1_score(y_va, model.predict(X_va), average="macro")

## 7. Reference baseline (your original notebook)
Naive Bayes with the default `CountVectorizer`. The default token pattern `\b\w\w+\b` **silently drops all emojis, punctuation and one-character tokens**, which are exactly the cues the assignment asks us to test. It is kept as the reference row.

In [ ]:
!pip install cloudpickle

# Override save_model locally to fix the joblib.externals.cloudpickle ImportError
def save_model(obj, name):
    import cloudpickle
    p = DIRS["models"] / f"{slug(name)}.pkl"
    with open(p, "wb") as f:
        cloudpickle.dump(obj, f)
    print(f"saved {p}  ({p.stat().st_size/1e6:.1f} MB)")

nb_ref = Pipeline([("vec", CountVectorizer()), ("clf", MultinomialNB())]).fit(X_tr, y_tr)
print("Validation macro-F1 (NB + default CountVectorizer): %.4f" % val_f1(nb_ref))
log_test("NB + CountVectorizer (default tokenizer)", nb_ref.predict(X_te), val_f1(nb_ref),
         nb_ref.predict_proba(X_te)[:, 1])
save_model(nb_ref, "nb_countvectorizer_reference")

## 8. Classical TF-IDF models and ablation study
All ablations use validation macro-F1. Differences below about 1 point are within split noise, so interpret small gaps with care.

* **Stage 1 - preprocessing ablation:** normalization, elongation handling, emojis, punctuation, hashtags and code-mixed (Latin) words.
* **Stage 2 - feature ablation:** word n-grams vs character n-grams vs both.
* **Stage 3 - classifier choice:** Logistic Regression vs Linear SVM.

In [ ]:
from joblib import Parallel, delayed

BASE = dict(normalize=True, elong=True)
CONFIGS = {
    "A minimal (no Urdu normalization)": dict(normalize=False, elong=False),
    "B + Urdu normalization":            dict(normalize=True,  elong=False),
    "C + elongation/laughter flags":     dict(),
    "D C, emojis removed":               dict(emo="remove"),
    "E C, punctuation removed":          dict(punct="remove"),
    "F C, emojis + punctuation removed": dict(emo="remove", punct="remove"),
    "G C, emojis grouped by sentiment":  dict(emo="group"),
    "H C, Latin (code-mixed) removed":   dict(latin="remove"),
    "I C, hashtags dropped":             dict(hashtag="drop"),
}
def cfg(kw): return {**BASE, **kw}

def make_vectorizer(prep, features="word+char", word_ngram=(1, 2)):
    word = TfidfVectorizer(preprocessor=prep, token_pattern=KEEP, ngram_range=word_ngram,
                           min_df=2, sublinear_tf=True)
    char = TfidfVectorizer(preprocessor=prep, analyzer="char_wb", ngram_range=(2, 5),
                           min_df=3, sublinear_tf=True, max_features=300_000)
    return {"word": word, "char": char, "word+char": make_union(word, char)}[features]

def make_clf(kind="lr", C=3.0):
    if kind == "lr":  return LogisticRegression(C=C, max_iter=3000)
    if kind == "svm": return LinearSVC(C=C)
    raise ValueError(kind)

def build(prep_kw, features="word+char", clf="lr", C=3.0, word_ngram=(1, 2)):
    return make_pipeline(make_vectorizer(make_prep(**prep_kw), features, word_ngram), make_clf(clf, C))

def _fit_score(name, kw, feats):
    m = build(cfg(kw), feats).fit(X_tr, y_tr)
    return {"preprocessing": name, "features": feats, "val_macro_F1": val_f1(m)}

# Stage 1a: screen every switch with fast word features, all configs in parallel
# (if Parallel raises a pickling error on your machine, set n_jobs=1)
screen = Parallel(n_jobs=-1)(delayed(_fit_score)(n, kw, "word") for n, kw in CONFIGS.items())
abl1_word = pd.DataFrame(screen).sort_values("val_macro_F1", ascending=False).reset_index(drop=True)
print("Stage 1a - all switches, word features:"); display(abl1_word.round(4))

# Stage 1b: confirm the 3 best (plus the 'all cues removed' ablation) with word+char features
confirm = list(dict.fromkeys(list(abl1_word.preprocessing.head(3)) + ["C + elongation/laughter flags",
                                                                      "F C, emojis + punctuation removed"]))
conf = Parallel(n_jobs=2)(delayed(_fit_score)(n, CONFIGS[n], "word+char") for n in confirm)
abl1 = pd.DataFrame(conf).sort_values("val_macro_F1", ascending=False).reset_index(drop=True)
print("\nStage 1b - confirmation with word+char features:"); display(abl1.round(4))
BEST_PREP_NAME = abl1.preprocessing.iloc[0]
BEST_PREP_KW = cfg(CONFIGS[BEST_PREP_NAME])
print("\nBest preprocessing:", BEST_PREP_NAME, BEST_PREP_KW)
save(abl1_word, "ablation_stage1_word.csv"); save(abl1, "ablation_stage1_wordchar.csv")

In [ ]:
# ---- Stage 2: feature ablation (best preprocessing)
rows = []
for feats, wng in [("word", (1, 1)), ("word", (1, 2)), ("word", (1, 3)), ("char", None), ("word+char", (1, 2))]:
    m = build(BEST_PREP_KW, feats, "lr", 3.0, wng or (1, 2)).fit(X_tr, y_tr)
    rows.append({"features": feats + (f" {wng}" if wng else " (2-5 char_wb)"), "val_macro_F1": val_f1(m)})
abl2 = pd.DataFrame(rows).sort_values("val_macro_F1", ascending=False).reset_index(drop=True)
display(abl2.round(4)); save(abl2, "ablation_stage2_features.csv")

# ---- Stage 3: classifier / regularization
rows = []
for clf, Cs in [("lr", [1, 3, 10]), ("svm", [0.1, 0.3, 1.0])]:
    for C in Cs:
        m = build(BEST_PREP_KW, "word+char", clf, C).fit(X_tr, y_tr)
        rows.append({"classifier": clf, "C": C, "val_macro_F1": val_f1(m)})
abl3 = pd.DataFrame(rows).sort_values("val_macro_F1", ascending=False).reset_index(drop=True)
display(abl3.round(4)); save(abl3, "ablation_stage3_classifier.csv")
BEST_CLF, BEST_C = abl3.classifier.iloc[0], float(abl3.C.iloc[0])

## 9. Urdu feature engineering (bonus): cue features + sarcasm phrase lexicon




In [ ]:
SARCASM_LEXICON = [
    "واہ", "واہ واہ", "کیا بات", "بہت خوب", "ماشاءاللہ", "ماشاء اللہ", "سبحان اللہ", "شکریہ",
    "مبارک", "تبدیلی", "کیوں نہیں", "بالکل", "جی ہاں", "ہاں جی", "ہاں ہاں", "اچھا", "لگتا ہے",
    "ویسے", "ذرا", "شاباش", "زبردست", "کمال", "بھئی", "اللہ کا شکر", "ایمانداری", "عظیم",
    "کون سا", "کیا خوب", "مہربانی", "خدا کا واسطہ", "نوٹس", "قوم", "دیکھو", "سن لو",
]
LEX_PREP = make_prep(normalize=True, elong=False, emo="remove", punct="remove")
LEX_NORM = [LEX_PREP(p) for p in SARCASM_LEXICON]
save(pd.DataFrame({"phrase": SARCASM_LEXICON, "normalized": LEX_NORM}), "sarcasm_lexicon.csv")

class CueFeatures(BaseEstimator, TransformerMixin):
    # hand-crafted sarcasm cues + lexicon flags -> scaled sparse matrix
    def __init__(self, use_lexicon=True):
        self.use_lexicon = use_lexicon
    def _raw(self, X):
        rows = []
        for t in X:
            el = emoji.emoji_list(t)
            stripped = t.rstrip()
            ends_emoji = int(bool(el) and el[-1]["match_end"] >= len(stripped))
            ns = max(sum(not c.isspace() for c in t), 1)
            r = [len(el), sum(c in LAUGH_EMOJIS for c in t), t.count("!"),
                 t.count("؟") + t.count("?"), int(bool(QUOTE_RE.search(t))),
                 int(bool(ELONG.search(t))), int(bool(LAUGH.search(t))),
                 sum(c.isascii() and c.isalpha() for c in t) / ns,
                 np.log1p(len(t.split())), ends_emoji, int(bool(PUNCT_REP.search(t)))]
            if self.use_lexicon:
                s = " " + LEX_PREP(t) + " "
                hits = [int(" " + p + " " in s) for p in LEX_NORM]
                r += hits + [sum(hits)]
            rows.append(r)
        return np.array(rows, dtype=float)
    def fit(self, X, y=None):
        self.scaler_ = StandardScaler(with_mean=False).fit(self._raw(X))
        return self
    def transform(self, X):
        return sparse.csr_matrix(self.scaler_.transform(self._raw(X)))

def build_with_cues(prep_kw, clf, C, use_lexicon=True, features="word+char"):
    vec = make_vectorizer(make_prep(**prep_kw), features)
    return make_pipeline(make_union(vec, CueFeatures(use_lexicon)), make_clf(clf, C))

rows = []
m_plain = build(BEST_PREP_KW, "word+char", BEST_CLF, BEST_C).fit(X_tr, y_tr)
rows.append({"model": "TF-IDF only", "val_macro_F1": val_f1(m_plain)})
m_cue = build_with_cues(BEST_PREP_KW, BEST_CLF, BEST_C, use_lexicon=False).fit(X_tr, y_tr)
rows.append({"model": "TF-IDF + cue features", "val_macro_F1": val_f1(m_cue)})
m_lex = build_with_cues(BEST_PREP_KW, BEST_CLF, BEST_C, use_lexicon=True).fit(X_tr, y_tr)
rows.append({"model": "TF-IDF + cue features + phrase lexicon", "val_macro_F1": val_f1(m_lex)})
abl4 = pd.DataFrame(rows); display(abl4.round(4)); save(abl4, "ablation_cue_features_lexicon.csv")

In [ ]:
# ---- evaluate the final classical models once on TEST, and save each fitted pipeline ----
candidates = {
    "TF-IDF (word+char) LR/SVM": m_plain,
    "TF-IDF + cues":             m_cue,
    "TF-IDF + cues + lexicon":   m_lex,
}
best_classical = max(candidates, key=lambda k: val_f1(candidates[k]))     # chosen on VALIDATION macro-F1
print("Best classical model on validation:", best_classical)
for name, mdl in candidates.items():
    log_test(name, mdl.predict(X_te), val_f1(mdl), score_proba(mdl, X_te))
    save_model(mdl, name)
save_json({"best_preprocessing": BEST_PREP_NAME, "preprocessing_kwargs": BEST_PREP_KW,
           "classifier": BEST_CLF, "C": BEST_C, "best_classical_by_val": best_classical},
          DIRS["models"] / "classical_config.json")

In [ ]:
# Data-driven sarcastic phrases: log-odds of word bi/trigrams on TRAIN only
cvp = CountVectorizer(preprocessor=make_prep(**BEST_PREP_KW), token_pattern=KEEP,
                      ngram_range=(2, 3), min_df=8, binary=True)
Xc = cvp.fit_transform(X_tr)
pos = np.asarray(Xc[y_tr == 1].sum(0)).ravel(); neg = np.asarray(Xc[y_tr == 0].sum(0)).ravel()
a = 1.0
lo = np.log((pos + a) / (pos.sum() + 2*a)) - np.log((neg + a) / (neg.sum() + 2*a))
vocab_p = np.array(cvp.get_feature_names_out())
top = lambda idx: pd.DataFrame({"phrase": vocab_p[idx], "sarc_count": pos[idx], "non_count": neg[idx],
                                "log_odds": lo[idx].round(2)})
top_sarc = top(np.argsort(-lo)[:25]);  print("Most SARCASTIC phrases:");     display(top_sarc); save(top_sarc, "phrases_most_sarcastic.csv")
top_non = top(np.argsort(lo)[:25]);    print("Most NON-SARCASTIC phrases:"); display(top_non); save(top_non, "phrases_most_nonsarcastic.csv")

In [ ]:
# ---- ablation summary figure (saved) ----
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
panels = [(abl1_word.assign(label=abl1_word.preprocessing), "Stage 1a: preprocessing (word features)"),
          (abl2.assign(label=abl2.features),                "Stage 2: feature type"),
          (abl3.assign(label=abl3.classifier + " C=" + abl3.C.astype(str)), "Stage 3: classifier / C"),
          (abl4.assign(label=abl4.model),                   "Cue features + lexicon")]
for ax, (d, title) in zip(axes.ravel(), panels):
    d = d.sort_values("val_macro_F1")
    ax.barh(d.label, d.val_macro_F1, color="steelblue")
    ax.set_xlim(d.val_macro_F1.min() - 0.01, d.val_macro_F1.max() + 0.005)
    ax.set_title(title, fontsize=10); ax.set_xlabel("validation macro-F1")
    for y_, v in enumerate(d.val_macro_F1): ax.text(v, y_, f" {v:.3f}", va="center", fontsize=8)
save_fig("07_ablation_summary", fig=fig)

In [ ]:
# ---- decision threshold tuned on VALIDATION only (the sarcastic class is only ~41% of the data) ----
def tune_threshold(p_val, y_val):
    grid = np.linspace(0.20, 0.80, 61)
    f1s = [f1_score(y_val, (p_val >= t).astype(int), average="macro") for t in grid]
    return float(grid[int(np.argmax(f1s))]), float(max(f1s))

m = candidates[best_classical]
p_va_m, p_te_m = score_proba(m, X_va), score_proba(m, X_te)
thr, vf = tune_threshold(p_va_m, y_va)
print(f"best threshold on validation: {thr:.2f}  (val macro-F1 {val_f1(m):.4f} at 0.50 -> {vf:.4f} at {thr:.2f})")
log_test(f"{best_classical} + tuned threshold", (p_te_m >= thr).astype(int), vf, p_te_m)
save_json({"model": best_classical, "threshold": thr, "val_macro_F1_at_threshold": vf},
          DIRS["models"] / "classical_tuned_threshold.json")

## 10. Neural model: BiGRU with masked max+mean pooling (PyTorch)



In [ ]:
import torch, torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.utils.data import DataLoader, TensorDataset

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
GRU_HIST = {}

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def encode_split(prep, texts, vocab, max_len):
    ids, lens = [], []
    for t in texts:
        toks = [vocab.get(w, 1) for w in TOKEN_RE.findall(prep(t))][:max_len]   # truncate at end
        lens.append(max(len(toks), 1))
        ids.append(toks + [0] * (max_len - len(toks)))                         # pad at end
    return torch.tensor(ids), torch.tensor(lens)

class BiGRU(nn.Module):
    def __init__(self, V, E=200, H=128, drop=0.4):
        super().__init__()
        self.emb = nn.Embedding(V, E, padding_idx=0)
        self.gru = nn.GRU(E, H, batch_first=True, bidirectional=True)
        self.drop = nn.Dropout(drop)
        self.fc = nn.Linear(4 * H, 2)
    def forward(self, x, lens):
        e = self.drop(self.emb(x))
        packed = pack_padded_sequence(e, lens.cpu(), batch_first=True, enforce_sorted=False)
        out, _ = self.gru(packed)
        out, _ = pad_packed_sequence(out, batch_first=True, total_length=x.size(1))
        mask = (x != 0).unsqueeze(-1)
        mx = out.masked_fill(~mask, -1e9).max(1).values
        mean = (out * mask).sum(1) / mask.sum(1).clamp(min=1)
        return self.fc(self.drop(torch.cat([mx, mean], 1)))

def predict_gru(model, X, L, bs=256):
    model.eval(); probs = []
    with torch.no_grad():
        for i in range(0, len(X), bs):
            logits = model(X[i:i+bs].to(DEVICE), L[i:i+bs])
            probs.append(torch.softmax(logits, 1)[:, 1].cpu())
    return torch.cat(probs).numpy()

def run_gru(prep_kw, epochs=50, patience=3, bs=64, lr=2e-3, seed=SEED, tag="bigru"):
    set_seed(seed)
    prep = make_prep(**prep_kw)
    cnt = collections.Counter(w for t in X_tr for w in TOKEN_RE.findall(prep(t)))
    vocab = {"<pad>": 0, "<unk>": 1}
    for w, c in cnt.items():
        if c >= 2: vocab[w] = len(vocab)
    lens_tr = [len(TOKEN_RE.findall(prep(t))) for t in X_tr]
    max_len = int(np.clip(np.percentile(lens_tr, 95), 32, 128))
    Xtr, Ltr = encode_split(prep, X_tr, vocab, max_len)
    Xva, Lva = encode_split(prep, X_va, vocab, max_len)
    Xte, Lte = encode_split(prep, X_te, vocab, max_len)
    print(f"vocab={len(vocab)} max_len={max_len}")
    loader = DataLoader(TensorDataset(Xtr, Ltr, torch.tensor(y_tr)), batch_size=bs, shuffle=True)

    model = BiGRU(len(vocab)).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    lossf = nn.CrossEntropyLoss()
    best, best_state, bad = -1, None, 0
    history = []
    for ep in range(epochs):
        model.train(); tot = 0
        for xb, lb, yb in loader:
            opt.zero_grad()
            loss = lossf(model(xb.to(DEVICE), lb), yb.to(DEVICE))
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
            tot += loss.item() * len(xb)
        pv = predict_gru(model, Xva, Lva)
        f1 = f1_score(y_va, (pv > 0.5).astype(int), average="macro")
        print(f"epoch {ep+1:2d} | train loss {tot/len(Xtr):.4f} | val macro-F1 {f1:.4f}")
        history.append({"epoch": ep + 1, "train_loss": tot / len(Xtr), "val_macro_F1": f1})
        if f1 > best:
            best, bad = f1, 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            bad += 1
            if bad >= patience: break
    model.load_state_dict(best_state)
    # ---- save everything needed to reload this model ----
    mdir = DIRS["models"] / slug(tag); mdir.mkdir(exist_ok=True)
    torch.save(best_state, mdir / "model_state.pt")
    save_json(vocab, mdir / "vocab.json")
    save_json({"prep_kwargs": prep_kw, "max_len": max_len, "vocab_size": len(vocab), "embed_dim": 200, "hidden": 128,
               "dropout": 0.4, "best_val_macro_F1": best, "seed": seed}, mdir / "config.json")
    GRU_HIST[tag] = pd.DataFrame(history); save(GRU_HIST[tag], f"training_curve_{slug(tag)}.csv")
    pt = predict_gru(model, Xte, Lte)
    return best, (pt > 0.5).astype(int), pt

GRU_VARIANTS = {
    "BiGRU (cues kept)":    dict(normalize=True, elong=True),
    "BiGRU (cues removed)": dict(normalize=True, elong=True, emo="remove", punct="remove"),
}
gru_out = {}
for name, kw in GRU_VARIANTS.items():
    print("\n=== ", name)
    gru_out[name] = run_gru(kw, tag=name)
for name, (vf1, pred, prob) in gru_out.items():
    log_test(name, pred, vf1, prob)

# learning curves of both BiGRU variants (saved)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for name, h in GRU_HIST.items():
    axes[0].plot(h.epoch, h.train_loss, marker="o", label=name)
    axes[1].plot(h.epoch, h.val_macro_F1, marker="o", label=name)
axes[0].set_title("Train loss"); axes[1].set_title("Validation macro-F1")
for ax in axes: ax.set_xlabel("epoch"); ax.legend(fontsize=8)
save_fig("08_bigru_training_curves", fig=fig)


## 11. Transformer experiment: fine-tuned XLM-RoBERTa

XLM-R was pretrained on CC-100 which includes Urdu, so it can represent Urdu script and code-mixed Latin tokens, and its sentencepiece tokenizer keeps emojis. We feed lightly normalized text (NFKC, quote unification, URL/mention tokens) and **do not** strip emojis or punctuation. The tokenizer adds its own start/end tokens.

Settings: max length 128, lr 2e-5, batch 32, 50 epochs, linear warm-up/decay, fp16 on GPU. Best epoch chosen on validation macro-F1.
Use a GPU runtime (Colab: Runtime -> Change runtime type -> T4). Add other models to `TRANSFORMERS` (for example `bert-base-multilingual-cased`, `google/muril-base-cased`) to compare them.

In [ ]:
!pip install torch torchvision torchaudio
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
# Removed: from your_module import make_prep
RUN_TRANSFORMER = torch.cuda.is_available()      # set True to force CPU (very slow)
SAVE_TRANSFORMER_WEIGHTS = True                  # ~1 GB for XLM-R base; set False to save only metrics/predictions
TRANSFORMERS = {"XLM-R base (emojis kept)": "xlm-roberta-base"}
EMOJI_AS_TEXT = False                              # True: convert emojis to words with emoji.demojize
TF_PREP = make_prep(normalize=False, elong=False)

def tf_text(t):
    t = TF_PREP(t)
    return emoji.demojize(t, delimiters=(" ", " ")) if EMOJI_AS_TEXT else t

def run_transformer(model_name, epochs=20, lr=2e-5, bs=32, max_len=128, seed=SEED, tag=None):
    tag = tag or model_name
    set_seed(seed)
    tok = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2).to(DEVICE)

    def enc(texts): return tok([tf_text(t) for t in texts], truncation=True, max_length=max_len)["input_ids"]
    ids_tr, ids_va, ids_te = enc(X_tr), enc(X_va), enc(X_te)

    def batches(ids, y=None, shuffle=False):
        order = np.random.permutation(len(ids)) if shuffle else np.arange(len(ids))
        for i in range(0, len(ids), bs):
            sel = order[i:i+bs]
            b = tok.pad({"input_ids": [ids[j] for j in sel]}, return_tensors="pt")
            yield b["input_ids"].to(DEVICE), b["attention_mask"].to(DEVICE), (None if y is None else torch.tensor(y[sel]).to(DEVICE))

    def predict(ids):
        model.eval(); out = []
        with torch.no_grad():
            for xb, mb, _ in batches(ids):
                with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
                    lg = model(input_ids=xb, attention_mask=mb).logits
                out.append(torch.softmax(lg.float(), 1)[:, 1].cpu())
        return torch.cat(out).numpy()

    use_amp = DEVICE.type == "cuda"          # fp16 only on GPU (was inverted)
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    steps = epochs * int(np.ceil(len(ids_tr) / bs))
    sched = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)

    best, best_state = -1, None
    history = []
    for ep in range(epochs):
        model.train(); t0 = time.time(); tot = 0
        for xb, mb, yb in batches(ids_tr, y_tr, shuffle=True):
            opt.zero_grad()
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
                loss = model(input_ids=xb, attention_mask=mb, labels=yb).loss
            scaler.scale(loss).backward()
            scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            tot += loss.item() * len(yb)
        pv = predict(ids_va)
        f1 = f1_score(y_va, (pv > 0.5).astype(int), average="macro")
        print(f"epoch {ep+1} | train loss {tot/len(ids_tr):.4f} | val macro-F1 {f1:.4f} | {time.time()-t0:.0f}s")
        history.append({"epoch": ep + 1, "train_loss": tot / len(ids_tr), "val_macro_F1": f1, "seconds": time.time() - t0})
        if f1 > best:
            best = f1
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    mdir = DIRS["models"] / slug(tag); mdir.mkdir(exist_ok=True)
    save(pd.DataFrame(history), f"training_curve_{slug(tag)}.csv")
    save_json({"hf_model": model_name, "epochs": epochs, "lr": lr, "batch_size": bs, "max_len": max_len,
               "emoji_as_text": EMOJI_AS_TEXT, "best_val_macro_F1": best, "seed": seed}, mdir / "config.json")
    if SAVE_TRANSFORMER_WEIGHTS:
        model.save_pretrained(mdir / "model"); tok.save_pretrained(mdir / "model")     # reload with from_pretrained(...)
    pt = predict(ids_te)
    return best, (pt > 0.5).astype(int), pt

if RUN_TRANSFORMER:
    for name, mname in TRANSFORMERS.items():
        print("\n=== ", name)
        vf1, pred, prob = run_transformer(mname, tag=name)
        log_test(name, pred, vf1, prob)
else:
    print("No GPU detected - skipping the transformer. Switch to a GPU runtime or set RUN_TRANSFORMER = True.")
    save_json({"transformer": "NOT RUN - no GPU in this session"}, DIRS["results"] / "transformer_status.json")

## 12. Results: model comparison (test set, primary metric = macro-F1)

In [ ]:
res = pd.DataFrame(RESULTS).T
res.insert(0, "val_macro_F1", pd.Series(VAL))
res = res.sort_values("macro_F1", ascending=False)
display(res.round(4))
save(res.round(4).reset_index().rename(columns={"index": "model"}), "results_summary.csv")

fig, ax = plt.subplots(figsize=(8, 0.5 * len(res) + 1.5))
sns.barplot(x=res.macro_F1.values, y=res.index, ax=ax, color="steelblue")
ax.set_xlim(max(0.5, res.macro_F1.min() - 0.05), min(1.0, res.macro_F1.max() + 0.03))
ax.set_xlabel("Test macro-F1"); ax.bar_label(ax.containers[0], fmt="%.3f", padding=3)
save_fig("09_model_comparison_macro_f1")

n = len(PRED); ncols = min(3, n); nrows = int(np.ceil(n / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(4.6 * ncols, 4 * nrows))
for ax, name in zip(np.atleast_1d(axes).ravel(), res.index):
    cm = confusion_matrix(y_te, PRED[name], labels=[0, 1])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
                xticklabels=["non-sarc", "sarc"], yticklabels=["non-sarc", "sarc"])
    ax.set_title(name, fontsize=9); ax.set_xlabel("Predicted"); ax.set_ylabel("True")
for ax in np.atleast_1d(axes).ravel()[n:]:
    ax.axis("off")
save_fig("10_confusion_matrices_all_models")

# ---- one confusion-matrix PNG per model, all test predictions, and a json copy of the results ----
for name in res.index:
    fig, ax = plt.subplots(figsize=(3.6, 3.2))
    sns.heatmap(confusion_matrix(y_te, PRED[name], labels=[0, 1]), annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
                xticklabels=["non-sarc", "sarc"], yticklabels=["non-sarc", "sarc"])
    ax.set_title(name, fontsize=8); ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    save_fig(f"cm_{slug(name)}", fig=fig, show=False); plt.close(fig)

pred_df = te_df[["raw_row_id", "text", "label"]].copy()
for name in PRED:
    pred_df[f"pred__{slug(name)}"] = PRED[name]
    if name in PROB: pred_df[f"prob__{slug(name)}"] = PROB[name]
save(pred_df, "test_predictions_all_models.csv", sub="predictions")
save_json({"test_metrics": RESULTS, "val_macro_F1": VAL}, DIRS["results"] / "results_summary.json")


## 13. Error analysis
Errors of the model with the best **validation** macro-F1. Automatic tags (emoji, negation, code-mixing, elongation/spelling, question, quote, short post) help you find a spread of phenomena. Idioms, irony/context dependence and ambiguous labels need your own reading, so fill the `commentary` column.

In [ ]:
best_name = max(VAL, key=VAL.get)
print("Best model on validation:", best_name)
pred = PRED[best_name]

NEG_RE = re.compile(r"(?<!\w)(نہ|نہیں|نہیں|مت|کبھی|بغیر|نا)(?!\w)")
err = te_df.copy()
err["pred"] = pred
err["prob_sarc"] = PROB[best_name] if best_name in PROB else np.nan
err = err[err.label != err.pred].copy()
err["error_type"] = np.where((err.label == 0) & (err.pred == 1), "false positive", "false negative")
err["tags"] = err.text.apply(lambda t: ",".join(k for k, v in {
    "emoji": len(emojis_in(t)) > 0, "negation": bool(NEG_RE.search(t)),
    "code-mixed": bool(re.search(r"[A-Za-z]{2,}", t)),
    "elongation/spelling": bool(ELONG.search(t) or LAUGH.search(t)),
    "question": bool(re.search(r"[؟?]", t)), "quote": bool(QUOTE_RE.search(t)),
    "short": len(t.split()) < 5, "long": len(t.split()) > 40}.items() if v))
print(f"{len(err)} errors out of {len(te_df)} test posts ({len(err)/len(te_df)*100:.1f}%)")
print(err.error_type.value_counts().to_dict())
print(err.tags.str.get_dummies(",").sum().sort_values(ascending=False).to_dict())

# a spread of >= 15 errors: sample per tag first, then fill randomly
picked = []
for tag in ["emoji", "negation", "code-mixed", "elongation/spelling", "question", "quote", "short", "long"]:
    sub = err[err.tags.str.contains(tag, regex=False) & ~err.index.isin(picked)]
    picked += list(sub.sample(min(2, len(sub)), random_state=SEED).index)
rest = err[~err.index.isin(picked)]
picked += list(rest.sample(min(max(0, 16 - len(picked)), len(rest)), random_state=SEED).index)
sel = err.loc[picked, ["text", "label", "pred", "prob_sarc", "error_type", "tags"]].copy()
sel["commentary"] = ""          # <- write your linguistic explanation for each
save(sel, "error_analysis.csv")          # the 16 sampled errors, with an empty commentary column to fill in
save(err[["text", "label", "pred", "prob_sarc", "error_type", "tags"]], "error_analysis_all_errors.csv")
for i, (_, r) in enumerate(sel.iterrows(), 1):
    print(f"\n#{i} [{r.error_type}] true={r.label} pred={r.pred} p(sarc)={r.prob_sarc:.2f} tags={r.tags}")
    print("   ", r.text[:220].replace("\n", " | "))

### Error commentary (write your own after reading the printed errors)

For each of at least 10 errors, name the phenomenon and why the model failed. Useful categories:

| # | Phenomenon | What to say |
|---|---|---|
| 1 | Irony / context dependence | The literal words are positive or neutral and sarcasm needs world knowledge (politics, public figures) |
| 2 | Emoji ambiguity | Emojis such as 🙏 😇 😂 appear in both classes, so they give a weak signal alone |
| 3 | Negation | Negation flips meaning and bag-of-words features cannot model scope |
| 4 | Idiom / proverb | Fixed expressions whose meaning is not compositional |
| 5 | Spelling variation | Same word spelled several ways, so rare forms are unseen |
| 6 | Code-mixing | English or Roman-Urdu tokens outside the pretrained or training vocabulary |
| 7 | Ambiguous / noisy label | A human reader would plausibly pick the other label |

In [ ]:
# ---- manifest of everything that was saved, plus one zip to download ----
import shutil
files = sorted(p for p in OUT.rglob("*") if p.is_file())
manifest = pd.DataFrame({"folder": [p.parent.name for p in files], "file": [p.name for p in files],
                         "size_KB": [round(p.stat().st_size / 1024, 1) for p in files]})
save(manifest, "output_manifest.csv")
print(manifest.groupby("folder").agg(files=("file", "count"), size_MB=("size_KB", lambda s: round(s.sum() / 1024, 1))))
zip_path = shutil.make_archive("urdu_sarcasm_outputs", "zip", OUT)
print("Bundle:", zip_path, f"({Path(zip_path).stat().st_size/1e6:.1f} MB)")
try:                                    # Colab: download the bundle
    from google.colab import files as _gc; _gc.download(zip_path)
except ImportError:
    pass